# Geração de Amostras V7 — Adaptativa (Algoritmo DAC)

Notebook **autônomo** que gera o dataset supervisionado para o tabuleiro 9×7
(4×3 caixas) usando o algoritmo **DAC — Diversidade Adaptativa em Cascata**.

**Quatro princípios:**

1. **Profundidade adaptativa por tensão estrutural τ** — Minimax raso na
   abertura, profundo no endgame.
2. **Boltzmann sampling com T(t) decrescente** — explora aberturas, joga
   "ótimo" no endgame.
3. **30 snapshots por partida** — cada partida cobre todos os t∈[1,30].
4. **Sem faixas/quotas** — meta única: 500.000 estados distintos.

**Fundamentação completa**: ver `docs/jogo_pontinhos/geracao_dados_v7_adaptativo.md`.

**Saída:** `dados/profundidade_minmax_7_adaptativo/dataset_pequeno_NNNN.npz`,
estrutura V2:

| Campo | Shape | Dtype | Fase |
|---|---|---|---|
| `estados` | `(N, 9, 7)` | `int8` (`{0,1,8,9}`) | 1 |
| `qtd_tracos` | `(N,)` | `int8` | 1 |
| `score_jogada` | `(N, 31)` | `float32` | 1 |
| `depth_jogada` | `(N,)` | `int8` | 1 |
| `depth_geracao` | `(N,)` | `int8` | 1 |
| `melhor_jogada` | `(N,)` | `<U5` | 2 |
| `score_melhor_jogada` | `(N, 31)` | `float32` | 2 |
| `depth_melhor_jogada` | `(N,)` | `int8` | 2 |
| `labels_canonicos` | `(31,)` | `<U5` | 1 |

**Retomada**: o estado é reconstruído a partir do diretório de NPZ — sem
sidecar JSON. Se o processo for interrompido, basta re-executar.


In [1]:
# %load_ext autoreload
# %autoreload 2

import os
import sys
import time
import math
import concurrent.futures
import multiprocessing as mp
from collections import Counter
from pathlib import Path

import numpy as np

# Adiciona a raiz do repositorio ao sys.path
ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "gerador_dados").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Nao encontrei a raiz do repositorio (pasta 'gerador_dados').")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from gerador_dados.jogo_pontinhos.gerador_amostras_v7_pontinhos import (
    jogar_partida_completa,
    calcular_scores_v7,
    LABELS_CANONICOS,
    N_LABELS,
    ALTURA,
    LARGURA,
    SCORE_INDISPONIVEL,
    DEPTH_NAO_CALCULADO,
    ESTADOS_POR_PARTIDA,
)

print(f"ROOT = {ROOT}")
print(f"N_LABELS = {N_LABELS}")
print(f"ALTURA x LARGURA = {ALTURA} x {LARGURA}")
print(f"ESTADOS_POR_PARTIDA = {ESTADOS_POR_PARTIDA}")


ROOT = D:\Desenvolvimento\arena-sagaz\arena-sagaz-backend
N_LABELS = 31
ALTURA x LARGURA = 9 x 7
ESTADOS_POR_PARTIDA = 30


## 1. Configuração

Meta única: 500.000 estados distintos. A distribuição por número de traços
emerge naturalmente — sem cotas por faixa, sem rejection sampling.

**Profundidade da Fase 2 (`PROFUNDIDADE_MELHOR_JOGADA`)** é fixa e define
o nome do diretório de saída.


In [2]:
ALVO_DISTINTOS = 500_000
PROFUNDIDADE_MELHOR_JOGADA = 7   # Fase 2: Minimax(p=7) por estado unico
TAMANHO_LOTE = 5_000              # estados por NPZ
SEED_GLOBAL = 20260508

OUTPUT_DIR = ROOT / "dados" / f"profundidade_minmax_{PROFUNDIDADE_MELHOR_JOGADA}_adaptativo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_WORKERS = max(1, mp.cpu_count() - 2)   # Ryzen 7 5700X => 14 workers
QUEUE_DEPTH = NUM_WORKERS * 4              # quantos futuros manter no ar

print(f"OUTPUT_DIR = {OUTPUT_DIR}")
print(f"NUM_WORKERS = {NUM_WORKERS}")
print(f"ALVO_DISTINTOS = {ALVO_DISTINTOS:,}")


OUTPUT_DIR = D:\Desenvolvimento\arena-sagaz\arena-sagaz-backend\dados\profundidade_minmax_7_adaptativo
NUM_WORKERS = 14
ALVO_DISTINTOS = 500,000


## 2. Recuperação do estado a partir do diretório

Lê todos os NPZs já existentes para reconstruir:

- O **set global de hashes** (chave: `estados[i].tobytes()`).
- O **índice do último NPZ** — para continuar a numeração.

A retomada é idempotente: re-executar o notebook do começo continua de onde
parou, sem sidecar JSON.


In [3]:
def proximo_indice_npz(output_dir: Path) -> int:
    npzs = sorted(output_dir.glob("dataset_pequeno_*.npz"))
    if not npzs:
        return 0
    ultimo = npzs[-1].stem.split("_")[-1]
    try:
        return int(ultimo)
    except ValueError:
        return 0


def reconstruir_estado_global(output_dir: Path):
    """Le todos os NPZs e devolve (hashes_set, ultimo_idx)."""
    hashes: set[bytes] = set()
    npzs = sorted(output_dir.glob("dataset_pequeno_*.npz"))
    for path in npzs:
        try:
            d = np.load(path)
        except Exception as e:
            print(f"[WARN] NPZ corrompido ignorado: {path.name} ({e})")
            continue
        estados = d["estados"]
        for i in range(estados.shape[0]):
            hashes.add(estados[i].tobytes())
    ultimo_idx = proximo_indice_npz(output_dir)
    return hashes, ultimo_idx


HASHES_GLOBAIS, ULTIMO_NPZ_IDX = reconstruir_estado_global(OUTPUT_DIR)

print(f"NPZs existentes: {len(sorted(OUTPUT_DIR.glob('dataset_pequeno_*.npz')))}")
print(f"Estados distintos ja gerados: {len(HASHES_GLOBAIS):,}")
print(f"Ultimo indice NPZ: {ULTIMO_NPZ_IDX}")
print(f"Restantes para a meta: {max(0, ALVO_DISTINTOS - len(HASHES_GLOBAIS)):,}")


NPZs existentes: 0
Estados distintos ja gerados: 0
Ultimo indice NPZ: 0
Restantes para a meta: 500,000


## 3. Fase 1 — Geração de estados via DAC

Cada worker joga uma partida completa (do tabuleiro vazio ao terminal) e
retorna 30 snapshots — um por t∈[1,30]. Main:

1. Mantém uma fila de `QUEUE_DEPTH` partidas submetidas ao pool.
2. Para cada partida concluída: acumula os 30 snapshots no buffer
   (incluindo duplicatas), atualiza set global de hashes para contagem
   de distintos.
3. A cada `TAMANHO_LOTE = 5.000` snapshots, grava NPZ atomicamente
   (`*.tmp.npz` → `os.replace`).
4. **Stop** ao atingir `len(HASHES_GLOBAIS) >= ALVO_DISTINTOS`. Drena
   partidas em curso.

NPZs da Fase 1 já têm a estrutura completa, mas com `melhor_jogada = ""`,
`score_melhor_jogada = -1e9` em todas as posições e `depth_melhor_jogada = 0`
(sentinela). A Fase 2 sobrescreve esses três campos.


In [4]:
def salvar_npz_fase1(buffer, idx_npz: int) -> Path:
    """Grava um NPZ V2 com os campos da Fase 1 preenchidos.

    `buffer` e lista de tuplas (mat_bytes, qtd_tracos, score_jogada_bytes,
    depth_jogada, depth_geracao). Os campos da Fase 2 ficam como
    placeholder. Sobrescrita atomica via .tmp.npz + os.replace.
    """
    n = len(buffer)
    estados_arr = np.empty((n, ALTURA, LARGURA), dtype=np.int8)
    qtd_tracos_arr = np.empty(n, dtype=np.int8)
    score_jogada_arr = np.empty((n, N_LABELS), dtype=np.float32)
    depth_jogada_arr = np.empty(n, dtype=np.int8)
    depth_geracao_arr = np.empty(n, dtype=np.int8)

    for i, (mb, qt, sj, dj, dg) in enumerate(buffer):
        estados_arr[i] = np.frombuffer(mb, dtype=np.int8).reshape(ALTURA, LARGURA)
        qtd_tracos_arr[i] = qt
        score_jogada_arr[i] = np.frombuffer(sj, dtype=np.float32)
        depth_jogada_arr[i] = dj
        depth_geracao_arr[i] = dg

    melhor_jogada_arr = np.empty(n, dtype="<U5")
    melhor_jogada_arr[:] = ""
    score_melhor_jogada_arr = np.full((n, N_LABELS), SCORE_INDISPONIVEL, dtype=np.float32)
    depth_melhor_jogada_arr = np.full(n, DEPTH_NAO_CALCULADO, dtype=np.int8)
    labels_arr = np.array(LABELS_CANONICOS, dtype="<U5")

    nome = f"dataset_pequeno_{idx_npz:04d}.npz"
    caminho = OUTPUT_DIR / nome
    tmp = caminho.with_name(caminho.stem + ".tmp.npz")  # mantem sufixo .npz
    np.savez_compressed(
        tmp,
        estados=estados_arr,
        qtd_tracos=qtd_tracos_arr,
        score_jogada=score_jogada_arr,
        depth_jogada=depth_jogada_arr,
        depth_geracao=depth_geracao_arr,
        melhor_jogada=melhor_jogada_arr,
        score_melhor_jogada=score_melhor_jogada_arr,
        depth_melhor_jogada=depth_melhor_jogada_arr,
        labels_canonicos=labels_arr,
    )
    os.replace(tmp, caminho)
    return caminho


def fase1_gerar(hashes_globais, ultimo_idx):
    """Loop principal da Fase 1. Retorna o ultimo idx de NPZ gravado."""
    rng_main = np.random.default_rng(SEED_GLOBAL)
    buffer = []
    npz_idx_ref = [ultimo_idx]
    contador_partidas = 0

    def _flush_se_cheio():
        if len(buffer) >= TAMANHO_LOTE:
            npz_idx_ref[0] += 1
            cam = salvar_npz_fase1(buffer[:TAMANHO_LOTE], npz_idx_ref[0])
            del buffer[:TAMANHO_LOTE]
            return cam
        return None

    inicio = time.time()
    print(f"Fase 1: alvo={ALVO_DISTINTOS:,} distintos | workers={NUM_WORKERS} | "
          f"queue={QUEUE_DEPTH}")

    with concurrent.futures.ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futuros: dict = {}

        def _submeter():
            seed_w = int(rng_main.integers(0, 2**31 - 1))
            f = executor.submit(jogar_partida_completa, seed_w)
            futuros[f] = seed_w

        # Burst inicial
        for _ in range(QUEUE_DEPTH):
            _submeter()

        ultimo_log = time.time()
        while len(hashes_globais) < ALVO_DISTINTOS:
            done, _pending = concurrent.futures.wait(
                list(futuros.keys()),
                return_when=concurrent.futures.FIRST_COMPLETED,
            )
            for f in done:
                futuros.pop(f)
                snapshots = f.result()
                contador_partidas += 1
                for snap in snapshots:
                    mb = snap[0]
                    if mb not in hashes_globais:
                        hashes_globais.add(mb)
                    buffer.append(snap)
                # Multiplos flushes se o buffer encheu varias vezes
                cam = _flush_se_cheio()
                while cam is not None:
                    print(f"  NPZ salvo: {cam.name} | "
                          f"distintos={len(hashes_globais):,}/{ALVO_DISTINTOS:,} | "
                          f"partidas={contador_partidas:,}")
                    cam = _flush_se_cheio()
                if len(hashes_globais) >= ALVO_DISTINTOS:
                    break
                _submeter()

            agora = time.time()
            if agora - ultimo_log > 30:
                elapsed = agora - inicio
                taxa_dist = len(hashes_globais) / elapsed if elapsed > 0 else 0
                taxa_part = contador_partidas / elapsed if elapsed > 0 else 0
                restante = max(0, ALVO_DISTINTOS - len(hashes_globais))
                eta = (restante / taxa_dist) if taxa_dist > 0 else float("inf")
                print(f"  partidas={contador_partidas:,} ({taxa_part:5.1f}/s) | "
                      f"distintos={len(hashes_globais):,}/{ALVO_DISTINTOS:,} "
                      f"({taxa_dist:6.1f}/s) | ETA={eta/60:5.1f} min")
                ultimo_log = agora

        # Drena partidas em curso (gravamos os snapshots delas, nao desperdicio)
        print(f"\nMeta atingida ({len(hashes_globais):,}). Drenando "
              f"{len(futuros)} partidas em curso...")
        for f in concurrent.futures.as_completed(list(futuros.keys())):
            snapshots = f.result()
            contador_partidas += 1
            for snap in snapshots:
                mb = snap[0]
                if mb not in hashes_globais:
                    hashes_globais.add(mb)
                buffer.append(snap)
            cam = _flush_se_cheio()
            while cam is not None:
                print(f"  NPZ (drain) salvo: {cam.name}")
                cam = _flush_se_cheio()
        futuros.clear()

    # Flush residual
    if buffer:
        npz_idx_ref[0] += 1
        cam = salvar_npz_fase1(buffer, npz_idx_ref[0])
        print(f"\nNPZ residual salvo: {cam.name} ({len(buffer)} amostras)")
        buffer.clear()

    elapsed = time.time() - inicio
    print(f"\nFase 1 concluida em {elapsed/60:.1f} min.")
    print(f"  Partidas:        {contador_partidas:,}")
    print(f"  Estados gravados: ~{contador_partidas * ESTADOS_POR_PARTIDA:,}")
    print(f"  Distintos:       {len(hashes_globais):,}")
    print(f"  NPZs gravados:   {npz_idx_ref[0]}")
    return npz_idx_ref[0]


# Para o multiprocessing funcionar no Jupyter/Windows e necessario que o
# pool seja criado a partir do __main__ (ja somos):
ULTIMO_NPZ_IDX = fase1_gerar(HASHES_GLOBAIS, ULTIMO_NPZ_IDX)


Fase 1: alvo=500,000 distintos | workers=14 | queue=56
  partidas=41 (  1.4/s) | distintos=1,167/500,000 (  38.7/s) | ETA=214.8 min
  partidas=88 (  1.5/s) | distintos=2,436/500,000 (  40.5/s) | ETA=205.0 min
  partidas=133 (  1.5/s) | distintos=3,618/500,000 (  40.0/s) | ETA=206.6 min
  NPZ salvo: dataset_pequeno_0001.npz | distintos=4,501/500,000 | partidas=167
  partidas=182 (  1.5/s) | distintos=4,875/500,000 (  40.4/s) | ETA=204.3 min
  partidas=227 (  1.5/s) | distintos=6,031/500,000 (  39.8/s) | ETA=206.8 min
  partidas=268 (  1.5/s) | distintos=7,075/500,000 (  38.9/s) | ETA=211.4 min
  partidas=321 (  1.5/s) | distintos=8,404/500,000 (  39.0/s) | ETA=209.8 min
  NPZ salvo: dataset_pequeno_0002.npz | distintos=8,735/500,000 | partidas=334
  partidas=372 (  1.5/s) | distintos=9,671/500,000 (  39.4/s) | ETA=207.3 min
  partidas=416 (  1.5/s) | distintos=10,756/500,000 (  39.0/s) | ETA=208.9 min
  partidas=464 (  1.5/s) | distintos=11,935/500,000 (  38.9/s) | ETA=209.3 min
  NPZ s

## 4. Fase 2 — Cálculo da melhor jogada (Minimax p=7)

1. Varre todos os NPZs em `OUTPUT_DIR` e separa os que ainda não foram
   processados (heurística: `melhor_jogada[0] == ""`).
2. Coleta o set único de estados (chave = `bytes`) entre todos esses
   NPZs — duplicatas são processadas apenas uma vez.
3. Em paralelo: para cada estado único, calcula
   `(rotulo, scores_(31,), profundidade_usada)` via
   `melhor_jogada_com_scores(estado, profundidade=PROFUNDIDADE_MELHOR_JOGADA)`.
   Slots inválidos recebem `-1e9`.
4. Reescreve cada NPZ atomicamente: mantém os campos da Fase 1, substitui
   `melhor_jogada`, `score_melhor_jogada`, `depth_melhor_jogada`.


In [ ]:
def npz_processado(d) -> bool:
    """Heuristica: se o primeiro melhor_jogada e nao-vazio, ja processado."""
    mj = d["melhor_jogada"]
    return mj.size > 0 and mj[0] != ""


def coletar_estados_pendentes(output_dir: Path):
    """Retorna (estados_unicos: set[bytes], pendentes: list[Path])."""
    estados_unicos: set[bytes] = set()
    pendentes: list[Path] = []
    for path in sorted(output_dir.glob("dataset_pequeno_*.npz")):
        with np.load(path) as d:
            if npz_processado(d):
                continue
            pendentes.append(path)
            estados = d["estados"]
            for i in range(estados.shape[0]):
                estados_unicos.add(estados[i].tobytes())
    return estados_unicos, pendentes


def calcular_cache_scores(estados_unicos: set[bytes], profundidade: int) -> dict:
    """Roda Minimax(profundidade) em paralelo para cada estado unico."""
    cache: dict[bytes, tuple[str, np.ndarray, int]] = {}
    items = list(estados_unicos)
    n_total = len(items)
    if n_total == 0:
        return cache
    inicio = time.time()
    ultimo_log = inicio
    with concurrent.futures.ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futuros = {
            executor.submit(calcular_scores_v7, (eb, profundidade)): eb
            for eb in items
        }
        n_done = 0
        for f in concurrent.futures.as_completed(futuros):
            eb = futuros[f]
            cache[eb] = f.result()
            n_done += 1
            agora = time.time()
            if agora - ultimo_log > 15 or n_done == n_total:
                taxa = n_done / (agora - inicio) if agora > inicio else 0
                eta = (n_total - n_done) / taxa if taxa > 0 else float("inf")
                print(f"  scoring: {n_done:>7,}/{n_total:>7,} "
                      f"({taxa:6.2f}/s) | ETA: {eta/60:5.1f} min")
                ultimo_log = agora
    return cache


def reescrever_npz_com_scores(path: Path, cache: dict) -> None:
    with np.load(path) as d:
        estados = d["estados"]
        qtd_tracos = d["qtd_tracos"]
        score_jogada = d["score_jogada"]
        depth_jogada = d["depth_jogada"]
        depth_geracao = d["depth_geracao"]
        labels_can = d["labels_canonicos"]
    n = estados.shape[0]
    melhor_jogada_arr = np.empty(n, dtype="<U5")
    score_melhor_jogada_arr = np.full((n, N_LABELS), SCORE_INDISPONIVEL, dtype=np.float32)
    depth_melhor_jogada_arr = np.empty(n, dtype=np.int8)
    for i in range(n):
        rotulo, scores, depth = cache[estados[i].tobytes()]
        melhor_jogada_arr[i] = rotulo
        score_melhor_jogada_arr[i] = scores
        depth_melhor_jogada_arr[i] = depth

    tmp = path.with_name(path.stem + ".tmp.npz")  # mantem sufixo .npz
    np.savez_compressed(
        tmp,
        estados=estados,
        qtd_tracos=qtd_tracos,
        score_jogada=score_jogada,
        depth_jogada=depth_jogada,
        depth_geracao=depth_geracao,
        melhor_jogada=melhor_jogada_arr,
        score_melhor_jogada=score_melhor_jogada_arr,
        depth_melhor_jogada=depth_melhor_jogada_arr,
        labels_canonicos=labels_can,
    )
    os.replace(tmp, path)


def fase2_scoring():
    estados_unicos, pendentes = coletar_estados_pendentes(OUTPUT_DIR)
    print(f"NPZs pendentes de scoring: {len(pendentes)}")
    print(f"Estados unicos a processar: {len(estados_unicos):,}")
    if not pendentes:
        print("Nada a fazer.")
        return
    print(f"Calculando Minimax(p={PROFUNDIDADE_MELHOR_JOGADA}) em "
          f"{NUM_WORKERS} workers...")
    cache = calcular_cache_scores(estados_unicos, PROFUNDIDADE_MELHOR_JOGADA)
    print(f"\nReescrevendo {len(pendentes)} NPZs...")
    for i, path in enumerate(pendentes, 1):
        reescrever_npz_com_scores(path, cache)
        print(f"  [{i:>3}/{len(pendentes)}] {path.name}")
    print(f"\nFase 2 concluida.")


fase2_scoring()


NPZs pendentes de scoring: 152
Estados unicos a processar: 501,039
Calculando Minimax(p=7) em 14 workers...
  scoring:       1/501,039 (  0.00/s) | ETA: 13537209.9 min
  scoring:      59/501,039 (  0.04/s) | ETA: 231547.1 min
  scoring:      66/501,039 (  0.04/s) | ETA: 209015.5 min
  scoring:      81/501,039 (  0.05/s) | ETA: 171965.1 min
  scoring:      88/501,039 (  0.05/s) | ETA: 159995.7 min
  scoring:      95/501,039 (  0.06/s) | ETA: 149566.7 min
  scoring:     111/501,039 (  0.06/s) | ETA: 129143.7 min
  scoring:     126/501,039 (  0.07/s) | ETA: 115055.6 min
  scoring:     143/501,039 (  0.08/s) | ETA: 102290.5 min
  scoring:     154/501,039 (  0.09/s) | ETA: 95800.9 min
  scoring:     172/501,039 (  0.10/s) | ETA: 86584.8 min
  scoring:     185/501,039 (  0.10/s) | ETA: 81246.4 min
  scoring:     217/501,039 (  0.12/s) | ETA: 69885.3 min
  scoring:     243/501,039 (  0.13/s) | ETA: 63008.1 min
  scoring:     246/501,039 (  0.13/s) | ETA: 62828.4 min
  scoring:     261/501,039

## 5. Auditoria final

Sanidade pós-pipeline:
- Domínio do tensor: cada `estado` deve ficar em `{0, 1, 8, 9}`.
- `melhor_jogada` não-vazio em todos os NPZs (exceto estados terminais).
- `score_melhor_jogada` por amostra: máximo finito.
- Distribuição empírica por `qtd_tracos` (deve ser bell-shaped).
- Distribuição empírica de `depth_jogada` (Fase 1, p adaptativo).
- Total de estados distintos ≥ 500.000.


In [ ]:
def auditoria(output_dir: Path) -> None:
    npzs = sorted(output_dir.glob("dataset_pequeno_*.npz"))
    print(f"NPZs encontrados: {len(npzs)}")
    if not npzs:
        return

    total = 0
    distintos: set[bytes] = set()
    valores_estado: set[int] = set()
    rotulos_vazios = 0
    estados_terminais = 0
    contador_t = Counter()
    contador_p_jogada = Counter()
    contador_p_geracao = Counter()
    contador_p_melhor = Counter()
    score_min = math.inf
    score_max = -math.inf

    for path in npzs:
        with np.load(path) as d:
            estados = d["estados"]
            qtd_tracos = d["qtd_tracos"]
            depth_jogada = d["depth_jogada"]
            depth_geracao = d["depth_geracao"]
            depth_melhor = d["depth_melhor_jogada"]
            melhor_jogada = d["melhor_jogada"]
            score_melhor = d["score_melhor_jogada"]
            n = estados.shape[0]
            total += n
            valores_estado.update(np.unique(estados).tolist())

            for i in range(n):
                distintos.add(estados[i].tobytes())
                contador_t[int(qtd_tracos[i])] += 1
                contador_p_jogada[int(depth_jogada[i])] += 1
                contador_p_geracao[int(depth_geracao[i])] += 1
                contador_p_melhor[int(depth_melhor[i])] += 1
                if melhor_jogada[i] == "":
                    rotulos_vazios += 1
                validos = score_melhor[i][score_melhor[i] > SCORE_INDISPONIVEL / 2]
                if validos.size == 0:
                    estados_terminais += 1
                else:
                    score_min = min(score_min, float(validos.min()))
                    score_max = max(score_max, float(validos.max()))

    print(f"\nTotal de amostras (incluindo duplicatas): {total:,}")
    print(f"Total de estados distintos: {len(distintos):,}")
    print(f"Valores unicos em 'estados': {sorted(valores_estado)}  (esperado: [0, 1, 8, 9])")
    print(f"Melhor jogada vazia: {rotulos_vazios:,}  (esperado ~ estados terminais)")
    print(f"Estados terminais: {estados_terminais}")
    print(f"score_melhor_jogada validos: min={score_min}, max={score_max}")

    print(f"\nDistribuicao por qtd_tracos (Fase 1):")
    if contador_t:
        max_v = max(contador_t.values())
        for t in sorted(contador_t):
            bar = "#" * int(contador_t[t] * 50 / max_v)
            print(f"  t={t:>2}: {contador_t[t]:>7,} {bar}")

    print(f"\nDistribuicao por depth_jogada (Fase 1, p adaptativo):")
    for p in sorted(contador_p_jogada):
        print(f"  p={p}: {contador_p_jogada[p]:>9,}  ({contador_p_jogada[p]/total*100:5.2f}%)")

    print(f"\nDistribuicao por depth_geracao (Fase 1):")
    for p in sorted(contador_p_geracao):
        print(f"  p={p}: {contador_p_geracao[p]:>9,}  ({contador_p_geracao[p]/total*100:5.2f}%)")

    print(f"\nDistribuicao por depth_melhor_jogada (Fase 2):")
    for p in sorted(contador_p_melhor):
        nome = "nao-calculado" if p == 0 else f"p={p}"
        print(f"  {nome}: {contador_p_melhor[p]:>9,}  ({contador_p_melhor[p]/total*100:5.2f}%)")


auditoria(OUTPUT_DIR)
